In [19]:
import numpy as np
import cvxpy as cp

In [20]:
# Projection onto a circle in R^2

def Proj_onto_circ(point, center, radius):
    x = cp.Variable(2)
    objective = cp.Minimize(cp.norm(x - point, 2))
    constraints = [cp.norm(x - center, 2) <= radius]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return x.value

In [21]:
# POCS

point = np.array([3, 7])

center1, radius1 = np.array([0, 0]), 1
center2, radius2 = np.array([2, 0]), 1.5

for _ in range(5):
    point = Proj_onto_circ(point, center1, radius1)
    point = Proj_onto_circ(point, center2, radius2)
    print(point)

[0.69811909 0.74505442]
[0.68811592 0.72729648]
[0.68753832 0.72625365]
[0.68750239 0.72618871]
[0.68750015 0.72618466]


In [22]:
# Dykstra

point = np.array([3, 7])

center1, radius1 = np.array([0, 0]), 1
center2, radius2 = np.array([2, 0]), 1.5

v = np.array([0, 0])
u = np.array([0, 0])

z = point

for _ in range(100):
    x = Proj_onto_circ(z + v, center1, radius1)
    v = z + v - x
    z = Proj_onto_circ(x + u, center2, radius2)
    u = x + u - z
    print(x, z)


[0.39391934 0.91914512] [0.69811909 0.74505442]
[0.43570419 0.90009003] [0.69957913 0.74759987]
[0.47150546 0.88186314] [0.70042252 0.74906499]
[0.50205758 0.8648342 ] [0.70077374 0.74967404]
[0.52807734 0.84919628] [0.70074496 0.74962415]
[0.550225   0.83501644] [0.70043301 0.74908321]
[0.56908654 0.82227764] [0.6999188  0.74819042]
[0.58516973 0.81091084] [0.69926806 0.74705853]
[0.59890772 0.80081805] [0.69853284 0.74577696]
[0.61066635 0.79188801] [0.69775345 0.74441517]
[0.6207528  0.78400636] [0.69696029 0.74302594]
[0.62942421 0.77706188] [0.69617572 0.74164837]
[0.63689561 0.77095006] [0.69541566 0.74031057]
[0.64334684 0.76557484] [0.6946909  0.73903194]
[0.64892858 0.76084933] [0.69400834 0.73782507]
[0.65376726 0.7566957 ] [0.69337187 0.73669734]
[0.65796928 0.75304477] [0.69278314 0.73565218]
[0.66162443 0.74983539] [0.69224219 0.73469011]
[0.66480868 0.74701367] [0.69174788 0.73380955]
[0.66758652 0.74453223] [0.6912983  0.73300745]
[0.67001286 0.74234949] [0.69089101 0.73

In [ ]:
# ADMM for Projection 

point = np.array([3, 7])

center1, radius1 = np.array([0, 0]), 1
center2, radius2 = np.array([2, 0]), 1.5

u = np.array([0, 0])
z = point
rho = 1

for _ in range(10):
    x = Proj_onto_circ((point + rho *(z - u))/(1+rho), center1, radius1)
    z = Proj_onto_circ(x + u, center2, radius2)
    u = u + x - z
    print(x, z)



[0.39391934 0.91914512] [0.69811909 0.74505442]
[0.46735515 0.88406972] [0.70024481 0.7487566 ]
[0.49493003 0.8689329 ] [0.70076782 0.74966382]
[0.51887643 0.85484932] [0.70082681 0.74976601]
[0.53967086 0.84187616] [0.70061589 0.7494004 ]
[0.55773454 0.83001946] [0.70021875 0.74871138]
[0.57343939 0.81924806] [0.6996912  0.74779479]
[0.58710988 0.80950732] [0.69907571 0.74672352]
[0.59902663 0.80072918] [0.69840561 0.74555488]
[0.60943093 0.79283924] [0.69770697 0.74433386]


In [25]:
# Poisson Equation (Set-up)

delta_x = 0.01
f = np.random.rand(99) * 300
c = 10
rho = 0.05

A = np.diag(np.ones(99), 0) + np.diag(np.ones(98)*-1,  -1)

A_lastrow = np.zeros(99)
A_lastrow[-1] = 1
A = np.vstack([A, A_lastrow]) 
print(A.shape)

# Remainder of the Poisson Equation
def F(u):
    return cp.sum_squares(u[:-1] - u[1:]) / delta_x + (u[0] ** 2)/ delta_x + (u[-1] ** 2) /delta_x - f @ u


(100, 99)


In [26]:
# Finding u_{k+1}

def Argmin_u_update(z, w):
    u = cp.Variable (99)
    objective = cp.Minimize(F(u) + rho / 2 * cp.norm(A@u - z + w, 2) ** 2)
    prob = cp.Problem(objective)
    prob.solve()
    return u.value


In [27]:
# Finding z_{k+1}

def Proj_onto_box(point):
    z = cp.Variable(100)
    objective = cp.Minimize(cp.norm(z - point, 2))
    constraints = [z >= -c * delta_x, z <= c * delta_x]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return z.value

In [29]:
# ADMM for Poisson Equation

w = np.zeros(100)
z = np.zeros(100)

for _ in range(100):
    u = Argmin_u_update(z, w)
    z = Proj_onto_box(A@u + w)
    w = w + A@u - z
    print(u[1], u[-1])

72.61309151014297 37.22833377440293
72.59504286600445 37.21907906450435
72.57694858160045 37.20980159113944
72.5588588886541 37.20052647135785
72.54077364783791 37.191253633777265
72.52269297195629 37.18198313646515
72.50461680613365 37.1727149517396
72.48654516965178 37.16344908939102
72.46847806320606 37.154185549729334
72.45041547856609 37.144924328526315
72.43235737206486 37.135665403395485
72.41430379629647 37.12640880130265
72.39625473266055 37.1171545127121
72.37821018007233 37.10790253706811
72.3601701374612 37.09865287382211
72.34213460373618 37.08940552241504
72.32410357779176 37.08016048228028
72.30607705851888 37.070917752849326
72.2880550448011 37.061677333549824
72.27003753554114 37.05243922381932
72.25202452956776 37.043203423057335
72.23401602572628 37.03396993067188
72.21601202283776 37.02473874605852
72.1980125208202 37.0155098691752
72.18001751594221 37.00628329810881
72.16202700886085 36.99705903319617
72.14404099993409 36.987837074620934
72.12605948307525 36.978617